# Medical RAG Chatbot -- MedQuAD + Gemini + FAISS

**Pipeline Overview:**
1. Install dependencies
2. Setup Gemini API
3. Load MedQuAD dataset from HuggingFace
4. Build FAISS vector index
5. RAG pipeline with LangChain + Gemini
6. Test queries
7. Visualizations (scatter plots, charts)
8. Launch Gradio UI

> **Note:** Get your free Gemini API key at https://aistudio.google.com

## Step 1 -- Install Dependencies

In [ ]:
!pip install -q langchain langchain-google-genai langchain-community
!pip install -q faiss-cpu
!pip install -q gradio
!pip install -q google-generativeai
!pip install -q datasets huggingface_hub
!pip install -q sentence-transformers
!pip install -q plotly scikit-learn umap-learn statsmodels
!pip install -q matplotlib nbformat
!pip install -q tiktoken

print('All packages installed successfully!')

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\manup\\AppData\\Roaming\\Python\\Python313\\site-packages\\fontTools\\cu2qu\\benchmark.py'
Check the permissions.



All packages installed successfully!


## Step 2 -- Gemini API Setup

In [14]:
import os
from pathlib import Path

# Try to load API key in order of preference:
# 1. GEMINI_API_KEY environment variable
# 2. GOOGLE_API_KEY environment variable
# 3. Local .env file
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')

if not GEMINI_API_KEY:
    env_file = Path('.env')
    if env_file.exists():
        try:
            import dotenv
        except ImportError as exc:
            raise ImportError(
                "python-dotenv is required to load a .env file. Install it with: pip install python-dotenv"
            ) from exc

        dotenv.load_dotenv(env_file)
        GEMINI_API_KEY = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')

if not GEMINI_API_KEY:
    raise ValueError(
        "Gemini API key not found. Set GEMINI_API_KEY (or GOOGLE_API_KEY) in your environment or .env file."
    )

GEMINI_API_KEY = GEMINI_API_KEY.strip()
os.environ['GOOGLE_API_KEY'] = GEMINI_API_KEY

import google.generativeai as genai
genai.configure(api_key=GEMINI_API_KEY)
print(f"Gemini configured. Key ends in: ...{GEMINI_API_KEY[-4:]}")

Gemini configured. Key ends in: ...gsRI


## Step 3 -- Load MedQuAD Dataset from HuggingFace

In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np

print('Loading MedQuAD dataset from HuggingFace...')
dataset = load_dataset('keivalya/MedQuad-MedicalQnADataset', split='train')

df = pd.DataFrame(dataset)
print(f'Dataset loaded! Total records: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head(3)

C:\Users\manup\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading MedQuAD dataset from HuggingFace...
Dataset loaded! Total records: 16407
Columns: ['qtype', 'Question', 'Answer']


,qtype,Question,Answer
0,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...
1,symptoms,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...
2,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,Individuals of all ages who come into contact ...


In [3]:
# Clean and prepare data
df.columns = [c.lower().strip() for c in df.columns]

q_col = [c for c in df.columns if 'question' in c or 'q_' in c][0]
a_col = [c for c in df.columns if 'answer' in c or 'a_' in c][0]
print(f'Question column: {q_col} | Answer column: {a_col}')

df = df[[q_col, a_col]].dropna()
df = df[df[a_col].str.len() > 30]
df = df.head(2000).reset_index(drop=True)
df.rename(columns={q_col: 'question', a_col: 'answer'}, inplace=True)

print(f'Cleaned dataset: {len(df)} Q&A pairs')
print(f'Avg question length: {df["question"].str.len().mean():.0f} chars')
print(f'Avg answer length:   {df["answer"].str.len().mean():.0f} chars')
df.head()

Question column: question | Answer column: answer
Cleaned dataset: 2000 Q&A pairs
Avg question length: 53 chars
Avg answer length:   1147 chars


,question,answer
0,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...
1,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...
2,Who is at risk for Lymphocytic Choriomeningiti...,Individuals of all ages who come into contact ...
3,How to diagnose Lymphocytic Choriomeningitis (...,"During the first phase of the disease, the mos..."
4,What are the treatments for Lymphocytic Chorio...,"Aseptic meningitis, encephalitis, or meningoen..."


## Step 4 -- Build FAISS Vector Index

In [8]:
try:
    from langchain_core.documents import Document
except ImportError:
    # Backward compatibility with older LangChain versions.
    from langchain.schema import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import time

documents = [
    Document(
        page_content=f"Question: {row['question']}\nAnswer: {row['answer']}",
        metadata={'index': i, 'question': row['question']}
    )
    for i, row in df.iterrows()
]
print(f'Created {len(documents)} LangChain documents')

print('Loading embedding model (first run ~1 min)...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'}
)
print('Embedding model loaded: all-MiniLM-L6-v2')

print('Building FAISS index...')
t0 = time.time()
vectorstore = FAISS.from_documents(documents, embeddings)
print(f'FAISS index built in {time.time()-t0:.1f}s')
print(f'Index size: {vectorstore.index.ntotal} vectors | Dims: {vectorstore.index.d}')

Created 2000 LangChain documents
Loading embedding model (first run ~1 min)...


C:\Users\manup\AppData\Local\Temp\ipykernel_22296\2107530167.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14281.43it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded: all-MiniLM-L6-v2
Building FAISS index...
FAISS index built in 27.2s
Index size: 2000 vectors | Dims: 384


In [26]:
import os

FAISS_PATH = './faiss_medquad_index'
vectorstore.save_local(FAISS_PATH)
print(f'FAISS index saved to: {FAISS_PATH}/')
print(f'Files: {os.listdir(FAISS_PATH)}')

test_query = 'What are the symptoms of diabetes?'
results = vectorstore.similarity_search(test_query, k=3)
print(f'\nTest search: "{test_query}"')
for i, r in enumerate(results, 1):
    print(f'  [{i}] {r.page_content[:100].replace(chr(10), " ")}...')

FAISS index saved to: ./faiss_medquad_index/
Files: ['index.faiss', 'index.pkl']

Test search: "What are the symptoms of diabetes?"
  [1] Question: What are the symptoms of Prevent diabetes problems: Keep your eyes healthy ? Answer: Often...
  [2] Question: What are the symptoms of Prevent diabetes problems: Keep your kidneys healthy ? Answer: In...
  [3] Question: What are the symptoms of Diabetic Neuropathies: The Nerve Damage of Diabetes ? Answer: Sym...


## Step 5 -- RAG Pipeline (LangChain + Gemini)

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI
try:
    from langchain.chains import RetrievalQA
except ImportError:
    from langchain_classic.chains import RetrievalQA

try:
    from langchain.prompts import PromptTemplate
except ImportError:
    from langchain_core.prompts import PromptTemplate

import google.generativeai as genai

MEDICAL_PROMPT = PromptTemplate(
    input_variables=['context', 'question'],
    template=(
        'You are a highly knowledgeable and empathetic medical assistant. '
        'Use ONLY the provided context from the MedQuAD medical database to answer the question. '
        'If the context does not contain enough information, say so and recommend a doctor.\n\n'
        'Guidelines:\n'
        '- Be precise, factual, and medically accurate\n'
        '- Use simple language when possible\n'
        '- Always remind the user to consult a doctor for personal advice\n\n'
        'Context from MedQuAD:\n{context}\n\n'
        'Patient Question: {question}\n\n'
        'Medical Answer:'
    )
)

def discover_gemini_models():
    models = []
    try:
        for m in genai.list_models():
            methods = set(getattr(m, 'supported_generation_methods', []) or [])
            if 'generateContent' not in methods:
                continue
            name = getattr(m, 'name', '')
            if name.startswith('models/'):
                name = name.split('/', 1)[1]
            if 'gemini' in name:
                models.append(name)
    except Exception:
        pass
    return models

preferred = [
    'gemini-2.5-flash',
    'gemini-2.0-flash',
    'gemini-2.5-pro',
    'gemini-1.5-flash',
    'gemini-1.5-pro',
]

discovered = discover_gemini_models()
candidate_models = [m for m in preferred if m in discovered] + [m for m in discovered if m not in preferred]

# Fallback list in case model discovery is unavailable.
if not candidate_models:
    candidate_models = [
        'gemini-2.5-flash',
        'gemini-2.0-flash',
        'gemini-1.5-flash',
    ]

llm = None
selected_model = None
last_error = None

for model_name in candidate_models:
    try:
        candidate_llm = ChatGoogleGenerativeAI(
            model=model_name,
            google_api_key=GEMINI_API_KEY,
            temperature=0.2,
            max_output_tokens=1024
        )
        _ = candidate_llm.invoke('Reply with: OK')
        llm = candidate_llm
        selected_model = model_name
        break
    except Exception as exc:
        last_error = exc

if llm is None:
    raise RuntimeError(
        f'No compatible Gemini model found. Last error: {last_error}'
    )

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4}
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    chain_type_kwargs={'prompt': MEDICAL_PROMPT},
    return_source_documents=True
)

print('RAG pipeline ready!')
print(f'  LLM: {selected_model}')
print('  Retriever: FAISS top-4')
print('  Prompt: Custom Medical Template')

RAG pipeline ready!
  LLM: gemini-2.5-flash
  Retriever: FAISS top-4
  Prompt: Custom Medical Template


## Step 6 -- Test Queries

In [28]:
def ask_medical_question(query, verbose=True):
    result = qa_chain.invoke({'query': query})
    answer = result['result']
    sources = result.get('source_documents', [])
    if verbose:
        print('=' * 70)
        print(f'QUESTION: {query}')
        print('=' * 70)
        print(f'ANSWER:\n{answer}')
        print('\nSOURCE DOCUMENTS:')
        for i, doc in enumerate(sources[:2], 1):
            print(f'  [{i}] {doc.metadata.get("question", "")[:80]}...')
        print('=' * 70 + '\n')
    return answer, sources

ans1, src1 = ask_medical_question('What are the main symptoms and risk factors of type 2 diabetes?')

QUESTION: What are the main symptoms and risk factors of type 2 diabetes?
ANSWER:
Many people with type 2 diabetes have no visible signs or symptoms, or their symptoms can be so mild that they might not be noticed. More than 5 million people in the United States have type 2 diabetes and do not know it.

The main symptoms of type 2 diabetes can include:
*   Increased thirst
*   Increased hunger
*   Fatigue
*   Increased urination, especially at night
*   Unexplained weight loss
*   Blurred vision
*   Sores that do not heal

The main risk factors for developing type 2 diabetes include:
*   Age 45 or older
*   Being overweight or obese
*   Being physically inactive
*   Having a parent or sibling with diabetes
*   Having a family background that is African American, Alaska Native, American Indian, Asian American, Hispanic/Latino, or Pacific Islander American
*   A history of giving birth to a baby weighing more than 9 pounds
*   A history of gestational diabetes (diabetes during pregnancy)

In [29]:
ans2, src2 = ask_medical_question('How is hypertension diagnosed and what lifestyle changes help manage it?')

QUESTION: How is hypertension diagnosed and what lifestyle changes help manage it?
ANSWER:
Hypertension, or high blood pressure, is diagnosed by a health care provider when multiple blood pressure tests, often repeated over several visits, consistently show a systolic blood pressure above 140 or a diastolic blood pressure consistently above 90.

Lifestyle changes that help manage high blood pressure include:

*   **Healthy Eating:**
    *   Follow a healthy eating plan like the Dietary Approaches to Stop Hypertension (DASH) eating plan, which focuses on fruits, vegetables, whole grains, and other heart-healthy foods lower in sodium.
    *   The DASH plan is low in fat and cholesterol, features fat-free or low-fat milk and dairy products, fish, poultry, and nuts, and suggests less red meat, sweets, added sugars, and sugar-containing beverages.
    *   Limit alcohol intake to two drinks per day for men and one per day for women.
    *   If you have kidney disease, a dietitian may recomme

In [30]:
ans3, src3 = ask_medical_question('What triggers asthma attacks and how are they treated?')

QUESTION: What triggers asthma attacks and how are they treated?
ANSWER:
I am sorry, but the provided context from the MedQuAD medical database does not contain information about what triggers asthma attacks or how they are treated.

For personalized medical advice regarding asthma, its triggers, and treatment options, please consult a doctor.

SOURCE DOCUMENTS:
  [1] What are the treatments for Sandhoff Disease ?...
  [2] What are the treatments for Motor Neuron Diseases ?...



## Step 7 -- Visualizations

In [4]:
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# -- Plot 1: Answer Length Distribution
fig1 = px.histogram(
    df, x=df['answer'].str.len(), nbins=60,
    title='Distribution of Answer Lengths in MedQuAD (2000 samples)',
    labels={'x': 'Answer Length (characters)', 'y': 'Count'},
    color_discrete_sequence=['#3B82F6'],
    template='plotly_white'
)
fig1.add_vline(
    x=df['answer'].str.len().median(), line_dash='dash', line_color='red',
    annotation_text=f'Median: {df["answer"].str.len().median():.0f}'
)
fig1.show()
print('Plot 1: Answer length distribution')

Plot 1: Answer length distribution


In [5]:
# -- Plot 2: Q vs A length scatter
df_plot = df.copy()
df_plot['q_len'] = df_plot['question'].str.len()
df_plot['a_len'] = df_plot['answer'].str.len().clip(upper=3000)

fig2 = px.scatter(
    df_plot.sample(500, random_state=42),
    x='q_len', y='a_len',
    title='Question Length vs Answer Length (500 samples)',
    labels={'q_len': 'Question Length (chars)', 'a_len': 'Answer Length (chars)'},
    color='a_len', color_continuous_scale='Viridis',
    opacity=0.65, template='plotly_white', trendline='ols'
)
fig2.show()
print('Plot 2: Q vs A scatter')

Plot 2: Q vs A scatter


In [9]:
# -- Plot 3: PCA 2D Embedding Visualization
from sklearn.decomposition import PCA

print('Computing embeddings for 300 samples...')
sample_df = df.sample(300, random_state=42).reset_index(drop=True)
sample_embs = embeddings.embed_documents(sample_df['question'].tolist())
emb_array = np.array(sample_embs)
print(f'Embeddings shape: {emb_array.shape}')

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(emb_array)
print(f'PCA variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%')

sample_df['length_bucket'] = pd.cut(
    sample_df['answer'].str.len(), bins=4,
    labels=['Short','Medium','Long','Very Long']
).astype(str)

fig3 = px.scatter(
    x=coords[:,0], y=coords[:,1],
    color=sample_df['length_bucket'],
    title='PCA 2D Projection of Medical Question Embeddings (300 samples)',
    labels={'x': 'PCA Component 1', 'y': 'PCA Component 2', 'color': 'Answer Length'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig3.update_traces(marker=dict(size=7, opacity=0.75))
fig3.show()
print('Plot 3: PCA embedding scatter')

Computing embeddings for 300 samples...
Embeddings shape: (300, 384)
PCA variance explained: 14.6%


Plot 3: PCA embedding scatter


In [10]:
# -- Plot 4: Medical Keyword Bar Chart
from collections import Counter
import re

KEYWORDS = [
    'diabetes','cancer','heart','hypertension','asthma',
    'infection','pain','treatment','symptoms','surgery',
    'medication','blood','disease','syndrome','disorder',
    'genetic','therapy','chronic','acute','immune'
]
all_text = ' '.join(df['question'].str.lower() + ' ' + df['answer'].str.lower())
counts = {kw: len(re.findall(r'\b' + kw + r'\b', all_text)) for kw in KEYWORDS}
kw_df = pd.DataFrame(counts.items(), columns=['Keyword','Count']).sort_values('Count')

fig4 = px.bar(
    kw_df, x='Count', y='Keyword', orientation='h',
    title='Top Medical Keyword Frequencies in MedQuAD',
    color='Count', color_continuous_scale='Blues',
    template='plotly_white'
)
fig4.show()
print('Plot 4: Keyword frequency')

Plot 4: Keyword frequency


In [11]:
# -- Plot 5: Cosine Similarity Heatmap
from sklearn.metrics.pairwise import cosine_similarity

test_queries_viz = [
    'symptoms of diabetes',
    'heart disease treatment',
    'asthma causes',
    'cancer prevention',
    'high blood pressure'
]
qembs = np.array(embeddings.embed_documents(test_queries_viz))
sim_mat = cosine_similarity(qembs)

fig5 = go.Figure(data=go.Heatmap(
    z=sim_mat, x=test_queries_viz, y=test_queries_viz,
    colorscale='RdBu', zmid=0.5,
    text=np.round(sim_mat, 2), texttemplate='%{text}'
))
fig5.update_layout(
    title='Cosine Similarity Heatmap Between Sample Medical Queries',
    template='plotly_white', height=450
)
fig5.show()
print('Plot 5: Cosine similarity heatmap')

Plot 5: Cosine similarity heatmap


In [12]:
# -- Plot 6: Retrieval Score Distribution
print('Computing retrieval scores for 50 random queries...')
sample_qs = df['question'].sample(50, random_state=7).tolist()
all_scores = []
for q in sample_qs:
    for _, score in vectorstore.similarity_search_with_score(q, k=4):
        all_scores.append(float(score))

fig6 = px.histogram(
    x=all_scores, nbins=40,
    title='FAISS Retrieval Distance Score Distribution (L2, lower = more similar)',
    labels={'x': 'L2 Distance', 'y': 'Count'},
    color_discrete_sequence=['#10B981'],
    template='plotly_white'
)
fig6.show()
print(f'Plot 6: Retrieval scores | Mean: {np.mean(all_scores):.3f}')

Computing retrieval scores for 50 random queries...


Plot 6: Retrieval scores | Mean: 0.632


## Step 8 -- Launch Gradio Chatbot UI

In [ ]:
import gradio as gr

SAMPLE_QS = [
    'What are the symptoms of type 2 diabetes?',
    'How is hypertension treated?',
    'What triggers asthma attacks?',
    'What is Alzheimer disease?',
    'Side effects of chemotherapy?'
]

def medical_chat(user_message, history):
    if not user_message.strip():
        return history
    try:
        result = qa_chain.invoke({'query': user_message})
        answer = result['result']
        sources = result.get('source_documents', [])
        if sources:
            refs = '\n'.join(
                f'- {doc.metadata.get("question","")[:60]}...'
                for doc in sources[:2]
            )
            answer += f'\n\n---\nSources from MedQuAD:\n{refs}'
    except Exception as e:
        answer = f'Error: {str(e)}'
    
    # Gradio 6.0 messages format
    history.append({"role": "user", "content": user_message})
    history.append({"role": "assistant", "content": answer})
    return history

CSS = '#chatbot { min-height: 450px; }'

with gr.Blocks(title='Medical RAG Chatbot') as demo:
    gr.HTML('<h1 style="text-align:center">Medical RAG Chatbot</h1>')
    gr.HTML('<p style="text-align:center;color:#6B7280">Gemini 2.5 Flash | FAISS | MedQuAD 2000 Q&A pairs</p>')
    gr.HTML('<div style="background:#FEF3C7;border-left:4px solid #F59E0B;padding:12px;border-radius:6px;color:#92400E;font-weight:500">'
            '<strong style="color:#92400E">⚠️ Disclaimer:</strong> This chatbot provides general medical information only. '
            'It is not a substitute for professional medical advice, diagnosis, or treatment. '
            'Always consult a qualified healthcare provider for personal health concerns.</div>')

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                elem_id='chatbot', label='Medical Assistant'
            )
            msg_box = gr.Textbox(
                placeholder='Ask a medical question...',
                label='Your Question', container=False
            )
            with gr.Row():
                send_btn = gr.Button('Send', variant='primary')
                clear_btn = gr.Button('Clear Chat', variant='secondary')

        with gr.Column(scale=1):
            gr.Markdown('### Sample Questions')
            for q in SAMPLE_QS:
                gr.Button(q, variant='secondary').click(
                    fn=lambda x=q: x, outputs=msg_box
                )

    send_btn.click(medical_chat, [msg_box, chatbot], [chatbot]).then(
        lambda: '', None, [msg_box]
    )
    msg_box.submit(medical_chat, [msg_box, chatbot], [chatbot]).then(
        lambda: '', None, [msg_box]
    )
    clear_btn.click(lambda: [], None, [chatbot])

demo.launch(share=True, debug=False, css=CSS, theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


## Step 9 -- Save Deployment Assets

In [18]:
import os, shutil

DEPLOY_DIR = './deployment_package'
os.makedirs(DEPLOY_DIR, exist_ok=True)

df.to_csv(f'{DEPLOY_DIR}/medquad_2000.csv', index=False)
shutil.copytree('./faiss_medquad_index', f'{DEPLOY_DIR}/faiss_index', dirs_exist_ok=True)

print('Deployment package contents:')
for item in os.listdir(DEPLOY_DIR):
    print(f'  {item}')

# Uncomment to save to Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copytree(DEPLOY_DIR, '/content/drive/MyDrive/MedRAG_Deploy', dirs_exist_ok=True)

print('All deployment assets saved!')

Deployment package contents:
  faiss_index
  medquad_2000.csv
All deployment assets saved!


Traceback (most recent call last):
  File "C:\Users\manup\AppData\Roaming\Python\Python313\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "C:\Users\manup\AppData\Roaming\Python\Python313\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "C:\Users\manup\AppData\Roaming\Python\Python313\site-packages\gradio\blocks.py", line 2169, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\manup\AppData\Roaming\Python\Python313\site-packages\gradio\blocks.py", line 1946, in postprocess_data
    prediction_value = block.postprocess(prediction_value)
  File 